### Extract the data from the csv

In [1]:
import numpy as np
from sklearn import preprocessing
raw_csv_data=np.loadtxt('Audiobooks_data (1).csv', delimiter=',')
unscaled_inputs_all=raw_csv_data[:, 1:-1]
targets_all=raw_csv_data[:, -1]

### Balance the dataset

In [2]:

num_one_targets=int(np.sum(targets_all))
zero_targets_counter=0
indices_to_remove=[]

for i in range(targets_all.shape[0]):
    if targets_all[i]==0:
        zero_targets_counter+=1
        if zero_targets_counter>num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_equal_priors=np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors=np.delete(targets_all, indices_to_remove, axis=0)
            

### Standardizing the inputs

In [3]:
scaled_inputs=preprocessing.scale(unscaled_inputs_equal_priors)

### Shuffle the data

In [4]:
shuffled_indices=np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs=scaled_inputs[shuffled_indices]
shuffled_targets=targets_equal_priors[shuffled_indices]

### Split the dataset into train, validation and test

In [5]:
samples_count=shuffled_inputs.shape[0]
train_samples_count=int(0.8*samples_count)
validation_samples_count=int(0.1*samples_count)
test_samples_count=samples_count-train_samples_count-validation_samples_count

train_inputs=shuffled_inputs[:train_samples_count]
train_targets=shuffled_targets[:train_samples_count]

validation_inputs=shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets=shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs=shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets=shuffled_targets[train_samples_count+validation_samples_count:]

print(np.sum(train_targets), train_samples_count, np.sum(train_targets)/train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets)/validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets)/test_samples_count)


1782.0 3579 0.4979044425817267
235.0 447 0.5257270693512305
220.0 448 0.49107142857142855


### Save the three datasets in *npz

In [6]:
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

### Create the machine learning algorithm

### Import the relevant libraries

In [7]:
import tensorflow as tf

C:\Users\HP\anaconda3\envs\tensorflow\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Data

In [9]:
npz=np.load('Audiobooks_data_train.npz')
train_inputs=npz['inputs'].astype(float)
train_targets=npz['targets'].astype(int)

npz=np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets=npz['inputs'].astype(float), npz['targets'].astype(int)

npz=np.load('Audiobooks_data_test.npz')
test_inputs, test_targets=npz['inputs'].astype(float), npz['targets'].astype(int)


### Model

In [14]:
input_size=10
output_size=2
hidden_layers_size=100
model=tf.keras.Sequential([
     tf.keras.layers.Dense(hidden_layers_size, activation='relu'),
     tf.keras.layers.Dense(hidden_layers_size, activation='relu'),
     tf.keras.layers.Dense(output_size, activation='softmax')

     
                          ])
earlystoping=tf.keras.callbacks.EarlyStopping(patience=2)

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['accuracy'])
batch_size=100
max_epochs=100
model.fit(train_inputs, train_targets, batch_size=batch_size, epochs=max_epochs, callbacks=[earlystoping], validation_data=[validation_inputs, validation_targets])

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7016 - loss: 0.5534 - val_accuracy: 0.7293 - val_loss: 0.4614
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7659 - loss: 0.4281 - val_accuracy: 0.7852 - val_loss: 0.4283
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7904 - loss: 0.3924 - val_accuracy: 0.7808 - val_loss: 0.3989
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8027 - loss: 0.3758 - val_accuracy: 0.8121 - val_loss: 0.3720
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8114 - loss: 0.3623 - val_accuracy: 0.7830 - val_loss: 0.3794
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8039 - loss: 0.3604 - val_accuracy: 0.7919 - val_loss: 0.3598
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8039 - loss: 0.3517 - val_accuracy: 0.7875 - val_loss: 0.3814
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8092 - loss: 0.3458 - val_accuracy: 0.7987 - 

In [16]:
test_loss, test_accuracy=model.evaluate(test_inputs, test_targets)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8170 - loss: 0.3177


In [22]:
print('\n Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100))


 Test loss: 0.32. Test accuracy: 81.70%
